# Análisis OLAP - MongoDB & Hive

Tres análisis sobre los datos del restaurante almacenados en MongoDB.  
Los resultados se persisten en Apache Hive como tablas permanentes.

| # | Análisis | Tabla Hive |
|---|---|---|
| 1 | Tendencias de consumo | `restaurantes.tendencias_consumo` |
| 2 | Horarios pico | `restaurantes.horarios_pico` |
| 3 | Crecimiento mensual | `restaurantes.crecimiento_mensual` |

## 0. Configuración

In [1]:
MONGO_URI = "mongodb://host.docker.internal:27017"
MONGO_DB  = "restaurantes_db"
HIVE_DB   = "restaurantes"

## 1. Sesión de Spark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("OLAP Restaurantes — MongoDB")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.4.0")
    .config("spark.mongodb.read.connection.uri", f"{MONGO_URI}/{MONGO_DB}")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} listo.")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eaa5fb41-73c8-4290-b11d-03503c6d469f;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;10.4.0 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
downloading https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.13/10.4.0/mongo-spark-connector_2.13-10.4.0.jar ...
	[SUCCESSFUL ] org.mongodb.spark#mongo-spark-connector_2.13;10.4.0!mongo-spark-connector_2.13.jar (623ms)
downloading 

Spark 4.1.2 listo.


## 2. Lectura de colecciones desde MongoDB

In [4]:
from pyspark.sql.functions import col

def read_col(collection):
    return (
        spark.read.format("mongodb")
        .option("collection", collection)
        .load()
        .withColumn("_id_str", col("_id").cast("string"))
    )

pedidos       = read_col("pedidos")
platos        = read_col("platos")
menus         = read_col("menus")
reservaciones = read_col("reservaciones")

print(f"pedidos:       {pedidos.count():,} documentos")
print(f"platos:        {platos.count():,} documentos")
print(f"menus:         {menus.count():,} documentos")
print(f"reservaciones: {reservaciones.count():,} documentos")

pedidos:       500 documentos
platos:        4,879 documentos
menus:         381 documentos
reservaciones: 200 documentos


## 3. Base de datos en Hive

In [5]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {HIVE_DB}")
spark.sql(f"USE {HIVE_DB}")
print(f"Base de datos '{HIVE_DB}' lista.")
spark.sql("SHOW DATABASES").show()

Base de datos 'restaurantes' lista.
+------------+
|   namespace|
+------------+
|     default|
|restaurantes|
+------------+



---
## Análisis 1 - Tendencias de consumo
Platos más pedidos y top 5 por categoría (excluyendo pedidos cancelados).

In [8]:
from pyspark.sql.functions import explode, sum as spark_sum, desc, row_number, col,rank
from pyspark.sql.window import Window

# 1. Extraer items de pedidos no cancelados
items = (
    pedidos
    .filter(col("estado") != "cancelado")
    .select(explode(col("items")).alias("item"))
    .select(
        col("item.plato_id").alias("plato_id"),
        col("item.cantidad").alias("cantidad")
    )
)

# 2. Joins para traer info de platos y menús
items_platos = items.join(
    platos.select(col("_id").alias("plato_id_join"), "nombre_plato", "precio", "menu_id"),
    items.plato_id == col("plato_id_join"), how="left"
)

items_platos_menus = items_platos.join(
    menus.select(col("_id").alias("menu_id_join"), col("nombre_menu").alias("categoria")),
    items_platos.menu_id == col("menu_id_join"), how="left"
)

# 3. Total global por nombre_plato (suma todos los restaurantes)
tendencias_total = (
    items_platos_menus
    .groupBy("nombre_plato")
    .agg(spark_sum("cantidad").alias("total_pedido"))
)

# 4. Total por restaurante para identificar el de mayor referencia
tendencias_por_restaurante = (
    items_platos_menus
    .groupBy("nombre_plato", "categoria", "precio")
    .agg(spark_sum("cantidad").alias("total_restaurante"))
)

# 5. Quedarse solo con el precio/categoría del restaurante que más vendió ese plato
w = Window.partitionBy("nombre_plato").orderBy(desc("total_restaurante"))

referencia = (
    tendencias_por_restaurante
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn", "total_restaurante")
)

# 6. Unir total con la referencia y ordenar
top_platos = (
    tendencias_total
    .join(referencia, on="nombre_plato", how="left")
    .orderBy(desc("total_pedido"))
)

print("Top 20 platos más pedidos:")
top_platos.show(20, truncate=False)

Top 20 platos más pedidos:


+------------------------------+------------+-----------+------+
|nombre_plato                  |total_pedido|categoria  |precio|
+------------------------------+------------+-----------+------+
|Enchiladas verdes vegetarianas|29          |Vegetariano|6729  |
|Refresco natural              |29          |Bebidas    |1551  |
|Bruschetta de tomate          |28          |Entradas   |3953  |
|Ensalada César                |28          |Entradas   |4147  |
|Yuca frita con chimichurri    |27          |Entradas   |3354  |
|Bagel con salmón ahumado      |27          |Desayunos  |7810  |
|Waffles con dulce de leche    |24          |Postres    |4060  |
|Cordero con hierbas           |23          |Cenas      |13448 |
|Pastel de chocolate           |23          |Postres    |3517  |
|Pancakes con miel             |23          |Desayunos  |4090  |
|Jugo verde detox              |22          |Bebidas    |2954  |
|Patacones con guacamole       |22          |Entradas   |3726  |
|Lomo fino en salsa borde

In [9]:
window_cat = Window.partitionBy("categoria").orderBy(desc("total_pedido"))

top_por_categoria = (
    top_platos
    .withColumn("rank", rank().over(window_cat))
    .filter(col("rank") <= 5)
    .orderBy("categoria", "rank")
)

print("Top 5 platos por categoría:")
top_por_categoria.show(50, truncate=False)

Top 5 platos por categoría:


+------------------------------+------------+---------+------+----+
|nombre_plato                  |total_pedido|categoria|precio|rank|
+------------------------------+------------+---------+------+----+
|Tamal de pollo                |21          |Almuerzos|4987  |1   |
|Casado vegetariano            |19          |Almuerzos|5843  |2   |
|Arroz con camarones           |18          |Almuerzos|8561  |3   |
|Burrito de frijoles y arroz   |17          |Almuerzos|5210  |4   |
|Casado de res                 |15          |Almuerzos|6920  |5   |
|Sopa de mondongo              |15          |Almuerzos|6018  |5   |
|Refresco natural              |29          |Bebidas  |1551  |1   |
|Jugo verde detox              |22          |Bebidas  |2954  |2   |
|Batido de fresa               |20          |Bebidas  |2243  |3   |
|Chocolate caliente            |20          |Bebidas  |2053  |3   |
|Jugo de naranja               |18          |Bebidas  |2124  |5   |
|Smoothie tropical             |18          |Beb

In [11]:
data = top_por_categoria.collect()
print(f"Filas recolectadas: {len(data)}")

schema = top_por_categoria.schema
df_final = spark.createDataFrame(data, schema)

df_final.write.mode("overwrite").saveAsTable(f"{HIVE_DB}.tendencias_consumo")
print("Guardado en Hive: tendencias_consumo")
spark.sql(f"SELECT * FROM {HIVE_DB}.tendencias_consumo LIMIT 5").show(truncate=False)

Filas recolectadas: 58


Guardado en Hive: tendencias_consumo
+--------------------+------------+---------+------+----+
|nombre_plato        |total_pedido|categoria|precio|rank|
+--------------------+------------+---------+------+----+
|Ensalada niçoise    |17          |Ensaladas|5602  |3   |
|Ensalada waldorf    |17          |Ensaladas|4884  |3   |
|Ensalada de atún    |14          |Ensaladas|4242  |5   |
|Bruschetta de tomate|28          |Entradas |3953  |1   |
|Ensalada César      |28          |Entradas |4147  |1   |
+--------------------+------------+---------+------+----+



---
## Análisis 2 - Horarios pico
Distribución de reservaciones activas y completadas por hora del día.

In [12]:
from pyspark.sql.functions import regexp_extract, count, asc

horarios_pico = (
    reservaciones
    .filter(col("estado").isin("activa", "completada"))
    .withColumn("hora", regexp_extract(col("hora_reservacion"), r"^(\d{2})", 1).cast("int"))
    .groupBy("hora")
    .agg(count("*").alias("total_reservaciones"))
    .orderBy(asc("hora"))
)

print("Reservaciones por hora del día:")
horarios_pico.show(24, truncate=False)

hora_pico = horarios_pico.orderBy(desc("total_reservaciones")).first()
print(f"Hora pico: {hora_pico['hora']}:00 con {hora_pico['total_reservaciones']} reservaciones")

Reservaciones por hora del día:
+----+-------------------+
|hora|total_reservaciones|
+----+-------------------+
|11  |13                 |
|12  |18                 |
|13  |15                 |
|14  |14                 |
|15  |16                 |
|16  |15                 |
|17  |13                 |
|18  |16                 |
|19  |14                 |
|20  |18                 |
|21  |16                 |
+----+-------------------+

Hora pico: 12:00 con 18 reservaciones


In [13]:
data = horarios_pico.collect()
df_final = spark.createDataFrame(data, horarios_pico.schema)
df_final.write.mode("overwrite").saveAsTable(f"{HIVE_DB}.horarios_pico")
print("Guardado en Hive: horarios_pico")
spark.sql(f"SELECT * FROM {HIVE_DB}.horarios_pico ORDER BY hora").show(truncate=False)

Guardado en Hive: horarios_pico
+----+-------------------+
|hora|total_reservaciones|
+----+-------------------+
|11  |13                 |
|12  |18                 |
|13  |15                 |
|14  |14                 |
|15  |16                 |
|16  |15                 |
|17  |13                 |
|18  |16                 |
|19  |14                 |
|20  |18                 |
|21  |16                 |
+----+-------------------+



---
## Análisis 3 - Crecimiento mensual
Volumen de pedidos agrupado por mes con variación porcentual mes a mes.

In [14]:
from pyspark.sql.functions import substring, lag, round as spark_round

crecimiento = (
    pedidos
    .withColumn("anio_mes", substring(col("fecha_orden"), 1, 7))
    .groupBy("anio_mes")
    .agg(count("*").alias("total_pedidos"))
    .orderBy(asc("anio_mes"))
)

print("Pedidos por mes:")
crecimiento.show(50, truncate=False)

Pedidos por mes:
+--------+-------------+
|anio_mes|total_pedidos|
+--------+-------------+
|2025-06 |26           |
|2025-07 |51           |
|2025-08 |43           |
|2025-09 |35           |
|2025-10 |39           |
|2025-11 |41           |
|2025-12 |47           |
|2026-01 |57           |
|2026-02 |29           |
|2026-03 |38           |
|2026-04 |43           |
|2026-05 |32           |
|2026-06 |19           |
+--------+-------------+



In [15]:
from pyspark.sql.functions import coalesce, lit

window_mes = Window.orderBy("anio_mes")
crecimiento_variacion = (
    crecimiento
    .withColumn("mes_anterior", lag("total_pedidos", 1).over(window_mes))
    .withColumn(
        "variacion_pct",
        coalesce(
            spark_round(
                ((col("total_pedidos") - col("mes_anterior")) / col("mes_anterior")) * 100, 2
            ),
            lit(0.0)
        )
    )
)
print("Variación porcentual mes a mes:")
crecimiento_variacion.show(50, truncate=False)

Variación porcentual mes a mes:


26/06/14 16:53:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 16:53:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 16:53:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 16:53:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 16:53:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 16:53:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 1

+--------+-------------+------------+-------------+
|anio_mes|total_pedidos|mes_anterior|variacion_pct|
+--------+-------------+------------+-------------+
|2025-06 |26           |NULL        |0.0          |
|2025-07 |51           |26          |96.15        |
|2025-08 |43           |51          |-15.69       |
|2025-09 |35           |43          |-18.6        |
|2025-10 |39           |35          |11.43        |
|2025-11 |41           |39          |5.13         |
|2025-12 |47           |41          |14.63        |
|2026-01 |57           |47          |21.28        |
|2026-02 |29           |57          |-49.12       |
|2026-03 |38           |29          |31.03        |
|2026-04 |43           |38          |13.16        |
|2026-05 |32           |43          |-25.58       |
|2026-06 |19           |32          |-40.63       |
+--------+-------------+------------+-------------+



In [17]:
data = crecimiento_variacion.collect()
df_final = spark.createDataFrame(data, crecimiento_variacion.schema)
df_final.write.mode("overwrite").saveAsTable(f"{HIVE_DB}.crecimiento_mensual")
print("Guardado en Hive: crecimiento_mensual")
spark.sql(f"SELECT * FROM {HIVE_DB}.crecimiento_mensual ORDER BY anio_mes").show(truncate=False)

Guardado en Hive: crecimiento_mensual
+--------+-------------+------------+-------------+
|anio_mes|total_pedidos|mes_anterior|variacion_pct|
+--------+-------------+------------+-------------+
|2025-06 |26           |NULL        |0.0          |
|2025-07 |51           |26          |96.15        |
|2025-08 |43           |51          |-15.69       |
|2025-09 |35           |43          |-18.6        |
|2025-10 |39           |35          |11.43        |
|2025-11 |41           |39          |5.13         |
|2025-12 |47           |41          |14.63        |
|2026-01 |57           |47          |21.28        |
|2026-02 |29           |57          |-49.12       |
|2026-03 |38           |29          |31.03        |
|2026-04 |43           |38          |13.16        |
|2026-05 |32           |43          |-25.58       |
|2026-06 |19           |32          |-40.63       |
+--------+-------------+------------+-------------+



---
## Resumen de tablas en Hive

In [18]:
print(f"Tablas en la base de datos '{HIVE_DB}':")
spark.sql(f"SHOW TABLES IN {HIVE_DB}").show(truncate=False)


Tablas en la base de datos 'restaurantes':
+------------+-------------------+-----------+
|namespace   |tableName          |isTemporary|
+------------+-------------------+-----------+
|restaurantes|crecimiento_mensual|false      |
|restaurantes|horarios_pico      |false      |
|restaurantes|tendencias_consumo |false      |
+------------+-------------------+-----------+



---
## Cerrar sesión

In [19]:
spark.stop()
print("Sesión cerrada.")

Sesión cerrada.
